# Uncertainty Propagation: From Parameters to Predictions

This notebook demonstrates how **parameter uncertainty propagates to prediction uncertainty**.

**Key concept**: Even after calibration, parameters remain uncertain. This uncertainty translates directly into uncertainty in model predictions. Honest modelling requires quantifying and communicating this uncertainty.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Patch
from IPython.display import HTML

# Import common setup
import sys
sys.path.append('../../_SUPPORT/src')

from calibration_common import (
    X_DOMAIN, L, H0, H1, K_TRUE, R_TRUE,
    X_OBS_STANDARD, H_OBS_STANDARD, H_TRUE_STANDARD,
    gw_model_1d, NOISE_STD
)

np.random.seed(42)

plt.rcParams['font.size'] = 11
plt.rcParams['figure.facecolor'] = 'white'

## The Setup: Same Model, Same Observations

We use the same 1D groundwater model and observations as the other demos:

$$h(x) = h_0 + (h_1 - h_0)\frac{x}{L} + \frac{R}{K} L \frac{x}{L}\left(1 - \frac{x}{L}\right)$$

In [ ]:
# Our observations (from calibration_common)
x_obs = X_OBS_STANDARD
h_obs = H_OBS_STANDARD

# True model
x_true, h_true = gw_model_1d(K_TRUE, R_TRUE)

print("Observations:")
for i, (x, h) in enumerate(zip(x_obs, h_obs)):
    print(f"  Well {i+1}: x = {x:.0f} m, h = {h:.2f} m")

print(f"\nTrue parameters: K = {K_TRUE} m/d, R = {R_TRUE} m/d")

## Calibrated Parameters with Uncertainty

After calibration, we don't get single "best" values - we get **probability distributions** reflecting our uncertainty.

For this example, assume calibration gave us:
- $K = 15 \pm 3$ m/d (20% coefficient of variation)
- $R = 0.001 \pm 0.00025$ m/d (25% CV)

In [ ]:
# Calibrated parameter distributions
K_mean, K_std = K_TRUE, K_TRUE * 0.20   # 20% CV
R_mean, R_std = R_TRUE, R_TRUE * 0.25   # 25% CV

print(f"Calibrated K: {K_mean:.1f} ± {K_std:.1f} m/d ({100*K_std/K_mean:.0f}% CV)")
print(f"Calibrated R: {R_mean:.5f} ± {R_std:.5f} m/d ({100*R_std/R_mean:.0f}% CV)")

## Monte Carlo Uncertainty Propagation

We sample from the parameter distributions and run the model many times to build up the **prediction uncertainty envelope**.

In [ ]:
def monte_carlo_ensemble(n_realizations, x, K_mean, K_std, R_mean, R_std):
    """
    Generate Monte Carlo ensemble of model predictions.
    
    Returns array of shape (n_realizations, len(x))
    """
    # Sample parameters (truncated to positive values)
    K_samples = np.maximum(K_mean + K_std * np.random.randn(n_realizations), 1.0)
    R_samples = np.maximum(R_mean + R_std * np.random.randn(n_realizations), 1e-6)
    
    # Run model for each realization
    ensemble = np.zeros((n_realizations, len(x)))
    for i in range(n_realizations):
        _, h = gw_model_1d(K_samples[i], R_samples[i], x=x)
        ensemble[i, :] = h
    
    return ensemble, K_samples, R_samples

# Generate ensemble
n_mc = 500
x_plot = np.linspace(0, L, 101)
ensemble, K_samples, R_samples = monte_carlo_ensemble(n_mc, x_plot, K_mean, K_std, R_mean, R_std)

print(f"Generated {n_mc} Monte Carlo realizations")
print(f"K range: {K_samples.min():.1f} to {K_samples.max():.1f} m/d")
print(f"R range: {R_samples.min():.5f} to {R_samples.max():.5f} m/d")

## Uncertainty Envelope Visualization

In [ ]:
def plot_uncertainty_envelope(x, ensemble, x_obs, h_obs, figsize=(10, 5)):
    """
    Plot prediction uncertainty envelope with percentiles.
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Calculate percentiles
    p5 = np.percentile(ensemble, 5, axis=0)
    p25 = np.percentile(ensemble, 25, axis=0)
    p50 = np.percentile(ensemble, 50, axis=0)
    p75 = np.percentile(ensemble, 75, axis=0)
    p95 = np.percentile(ensemble, 95, axis=0)
    
    # Plot uncertainty bands
    ax.fill_between(x, p5, p95, alpha=0.2, color='blue', label='90% interval')
    ax.fill_between(x, p25, p75, alpha=0.4, color='blue', label='50% interval')
    ax.plot(x, p50, 'b-', linewidth=2, label='Median prediction')
    
    # Plot observations
    ax.scatter(x_obs, h_obs, c='black', s=80, zorder=5, 
               marker='o', edgecolor='white', linewidth=1.5,
               label='Observations')
    
    # Labels
    ax.set_xlabel('Distance x [m]')
    ax.set_ylabel('Hydraulic head h [m]')
    ax.set_title('Prediction Uncertainty from Parameter Uncertainty', 
                fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, L)
    
    # Add annotation
    ax.text(0.02, 0.02, f'Based on {len(ensemble)} Monte Carlo realizations',
           transform=ax.transAxes, fontsize=9, color='gray')
    
    plt.tight_layout()
    return fig, ax

fig, ax = plot_uncertainty_envelope(x_plot, ensemble, x_obs, h_obs)
plt.show()

## Animated Uncertainty Buildup

Watch how the uncertainty envelope emerges as we add more Monte Carlo realizations.

In [ ]:
def create_uncertainty_animation(x, ensemble, x_obs, h_obs, K_samples, R_samples, figsize=(10, 5)):
    """
    Create animation showing uncertainty envelope building up.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={'width_ratios': [2, 1]})
    ax1, ax2 = axes
    
    # Fixed axis limits
    h_min = ensemble.min() - 0.5
    h_max = ensemble.max() + 0.5
    
    # Frame schedule: show more detail at start, then speed up
    n_at_frames = [1, 2, 3, 5, 10, 20, 50, 100, 200, len(ensemble)]
    n_at_frames = [n for n in n_at_frames if n <= len(ensemble)]
    if len(ensemble) not in n_at_frames:
        n_at_frames.append(len(ensemble))
    
    def animate(frame_idx):
        ax1.clear()
        ax2.clear()
        
        n = n_at_frames[min(frame_idx, len(n_at_frames)-1)]
        current_ensemble = ensemble[:n]
        
        # Left plot: Uncertainty envelope
        if n >= 3:
            p5 = np.percentile(current_ensemble, 5, axis=0)
            p25 = np.percentile(current_ensemble, 25, axis=0)
            p50 = np.percentile(current_ensemble, 50, axis=0)
            p75 = np.percentile(current_ensemble, 75, axis=0)
            p95 = np.percentile(current_ensemble, 95, axis=0)
            
            ax1.fill_between(x, p5, p95, alpha=0.2, color='blue')
            ax1.fill_between(x, p25, p75, alpha=0.4, color='blue')
            ax1.plot(x, p50, 'b-', linewidth=2)
        else:
            # Show individual lines for first few
            for i in range(n):
                ax1.plot(x, current_ensemble[i], 'b-', alpha=0.5, linewidth=1)
        
        # Plot latest realization in different color
        if n < 50:
            ax1.plot(x, current_ensemble[-1], 'r-', linewidth=1.5, alpha=0.7)
        
        ax1.scatter(x_obs, h_obs, c='black', s=80, zorder=5,
                   marker='o', edgecolor='white', linewidth=1.5)
        
        ax1.set_xlim(0, L)
        ax1.set_ylim(h_min, h_max)
        ax1.set_xlabel('Distance x [m]')
        ax1.set_ylabel('Hydraulic head h [m]')
        ax1.set_title(f'Prediction Uncertainty\n{n} realizations', fontsize=11, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Right plot: Parameter samples
        ax2.scatter(K_samples[:n], R_samples[:n]*1000, alpha=0.4, s=20, c='blue')
        if n < 50:
            ax2.scatter(K_samples[n-1], R_samples[n-1]*1000, c='red', s=60, zorder=5)
        ax2.axvline(K_mean, color='gray', linestyle='--', alpha=0.5)
        ax2.axhline(R_mean*1000, color='gray', linestyle='--', alpha=0.5)
        
        ax2.set_xlim(K_mean - 4*K_std, K_mean + 4*K_std)
        ax2.set_ylim((R_mean - 4*R_std)*1000, (R_mean + 4*R_std)*1000)
        ax2.set_xlabel('K [m/d]')
        ax2.set_ylabel('R [mm/d]')
        ax2.set_title('Parameter Samples', fontsize=11, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return []
    
    # Extra frames at the end to hold
    total_frames = len(n_at_frames) + 3
    anim = animation.FuncAnimation(fig, animate, frames=total_frames, 
                                   interval=600, blit=True)
    plt.close(fig)
    return anim, fig

anim, fig = create_uncertainty_animation(x_plot, ensemble, x_obs, h_obs, K_samples, R_samples)
HTML(anim.to_jshtml())

## The Key Insight: Prediction Uncertainty Varies Spatially

Notice that uncertainty is **not uniform** across the domain:
- Near the boundaries, head is constrained by boundary conditions
- In the middle, uncertainty is largest as parameter effects accumulate

In [ ]:
# Calculate prediction uncertainty at different locations
locations = [0, 250, 500, 750, 1000]

print("Prediction uncertainty by location:")
print("-" * 55)
print(f"{'Location [m]':>12} {'Mean [m]':>10} {'Std [m]':>10} {'95% CI [m]':>18}")
print("-" * 55)

for loc in locations:
    idx = np.argmin(np.abs(x_plot - loc))
    h_mean = np.mean(ensemble[:, idx])
    h_std = np.std(ensemble[:, idx])
    h_5 = np.percentile(ensemble[:, idx], 5)
    h_95 = np.percentile(ensemble[:, idx], 95)
    print(f"{loc:>12} {h_mean:>10.2f} {h_std:>10.3f} {f'[{h_5:.2f}, {h_95:.2f}]':>18}")

print("-" * 55)

## Save Animation as GIF

In [ ]:
# Recreate animation for saving
anim, fig = create_uncertainty_animation(x_plot, ensemble, x_obs, h_obs, K_samples, R_samples, 
                                         figsize=(10, 4.5))

# Save as GIF
gif_path = 'uncertainty_envelope.gif'
anim.save(gif_path, writer='pillow', fps=1.5, dpi=100)
print(f"Animation saved to {gif_path}")

# Display file size
import os
size_kb = os.path.getsize(gif_path) / 1024
print(f"File size: {size_kb:.0f} KB")

## Linking Sensitivity and Uncertainty

The prediction uncertainty depends on:
1. **Parameter sensitivity** - how much does the output change per unit change in parameter?
2. **Parameter uncertainty** - how uncertain is each parameter?

$$\sigma^2_{prediction} \approx \sum_i \left(\frac{\partial h}{\partial p_i}\right)^2 \cdot \sigma^2_{p_i}$$

This means:
- A **sensitive parameter with low uncertainty** → moderate prediction uncertainty
- An **insensitive parameter with high uncertainty** → low prediction uncertainty  
- A **sensitive parameter with high uncertainty** → HIGH prediction uncertainty (priority for data collection!)

## Key Takeaways

1. **Calibration doesn't eliminate uncertainty** - it reduces it, but uncertainty remains

2. **Parameter uncertainty → Prediction uncertainty** - Monte Carlo propagation quantifies this

3. **Report ranges, not point values** - "head will be 97-98 m" is more honest than "head = 97.52 m"

4. **Uncertainty varies spatially** - some locations are better constrained than others

5. **Reduce uncertainty through better data** - especially for sensitive parameters